# 01 · Data Preparation

Loads the [gretelai/synthetic_text_to_sql](https://huggingface.co/datasets/gretelai/synthetic_text_to_sql) dataset and converts it into MLX-compatible JSONL format (chat template).

**Output files:**
- `data/train.jsonl` — 5,000 training samples
- `data/valid.jsonl` — 500 validation samples
- `data/test.jsonl` — 500 test samples

Run this notebook once before training (`02_fine_tuning_full.ipynb`).


## Step 1 · Load & explore dataset

Inspect the schema and a sample record to confirm the dataset structure.


In [ ]:
from datasets import load_dataset

ds = load_dataset("gretelai/synthetic_text_to_sql")
print(f"Train: {len(ds['train'])} | Test: {len(ds['test'])}")
print(f"\nColumns: {ds['train'].column_names}")
print(f"\n--- Sample ---")
sample = ds['train'][0]
for k in ['sql_prompt', 'sql_context', 'sql', 'sql_explanation', 'sql_complexity']:
    print(f"\n[{k}]:\n{sample[k]}")


## Step 2 · Convert to MLX chat-template JSONL

Reformat each record into a `messages` array with `system`/`user`/`assistant` roles — the format `mlx_lm.lora` expects for instruction tuning.

We subsample 5,000 training / 500 validation / 500 test records for fast iteration on a laptop. The held-out test set is later used by `03_evaluation.ipynb`.


In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

SYSTEM_PROMPT = """You are a SQL expert. Given a database schema and a natural language question, generate the correct SQL query and a brief explanation.

Respond in this exact format:
SQL: <your sql query>
Explanation: <brief explanation>"""

def format_sample(sample):
    user_msg = f"""Database Schema:
{sample['sql_context']}

Question: {sample['sql_prompt']}"""

    assistant_msg = f"""SQL: {sample['sql']}
Explanation: {sample['sql_explanation']}"""

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg}
        ]
    }

train_data = ds['train'].shuffle(seed=42).select(range(5000))
val_data = ds['test'].shuffle(seed=42).select(range(500))
test_data = ds['test'].shuffle(seed=42).select(range(500, 1000))

for split_name, split_data in [("train", train_data), ("valid", val_data), ("test", test_data)]:
    output_path = DATA_DIR / f"{split_name}.jsonl"
    with open(output_path, "w") as f:
        for s in split_data:
            f.write(json.dumps(format_sample(s)) + "\n")
    print(f"Saved {len(split_data)} samples to {output_path}")

# Sanity check
with open(DATA_DIR / "train.jsonl") as f:
    first = json.loads(f.readline())
print("\n--- First training sample preview ---")
for msg in first["messages"]:
    print(f"\n[{msg['role']}]:\n{msg['content'][:200]}")
